In [ ]:
import pymysql
import pandas as pd

def read_from_mysql():
    # 数据库连接配置
    config = {
        'host': '10.0.21.126',      # 数据库地址
        'port': 3306,             # 端口
        'user': 'bigquery_user',           # 用户名
        'password': 'xdf#42sdfkjdfiyuISYFj76',   # 密码
        'database': 'niuniuh5_db',    # 数据库名
        'charset': 'utf8mb4',     # 字符集
        'cursorclass': pymysql.cursors.DictCursor
    }

    try:
        # 建立连接
        connection = pymysql.connect(**config)
        
        try:
            # 方法1：使用 cursor 执行 SQL
            with connection.cursor() as cursor:
                sql = "SELECT nPlayerId, nClubID, nAction, szTime, loginIp FROM table_web_loginlog LIMIT 10"
                cursor.execute(sql)
                result = cursor.fetchall()
                print("--- Cursor Result ---")
                for row in result:
                    print(row)

            # # 方法2：使用 pandas 读取 (推荐用于数据分析)
            # sql = "SELECT * FROM your_table LIMIT 10"
            # df = pd.read_sql(sql, connection)
            # print("\n--- Pandas DataFrame ---")
            # print(df.head())

        finally:
            # 关闭连接
            connection.close()
            
    except Exception as e:
        print(f"Error: {e}")

In [5]:
read_from_mysql()

--- Cursor Result ---
{'nPlayerId': 133134, 'nClubID': 666098, 'nAction': 1, 'szTime': datetime.datetime(2022, 11, 1, 10, 20, 28), 'loginIp': '113.118.234.183'}
{'nPlayerId': 133422, 'nClubID': 666639, 'nAction': 1, 'szTime': datetime.datetime(2022, 11, 1, 10, 24, 2), 'loginIp': '113.118.234.183'}
{'nPlayerId': 133343, 'nClubID': 666000, 'nAction': 1, 'szTime': datetime.datetime(2022, 11, 1, 10, 24, 2), 'loginIp': '113.118.234.183'}
Error: Execution failed on sql 'SELECT * FROM your_table LIMIT 10': (1146, "Table 'niuniuh5_db.your_table' doesn't exist")


/tmp/ipykernel_2763359/1642780382.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, connection)


In [ ]:
from pyhive import hive
import pandas as pd
from datetime import datetime, timedelta

# HiveServer2 连接参数
host = "10.0.21.126"  # HiveServer2 主机地址
port = 10000         # HiveServer2 端口
username = "root"    # 用户名
database = "default" # 指定数据库


# 建立连接
conn = hive.connect(
    host=host,
    port=port,
    username=username,
    database=database
    #auth='NOSASL'  # 适用于无认证模式
)

# 计算昨天日期
yesterday = datetime.now() - timedelta(days=1)
dt_str = yesterday.strftime('%Y-%m-%d')
print(f"Querying for date: {dt_str}")

# 创建游标
cursor = conn.cursor()

# 执行查询
# 汇总表：昨日游戏局数、昨日用户数（非机器人）、近一个月用户数、昨日新增用户数、
# 机器人游戏局数、机器人游戏占比、机器人胜率、发牌干预局数
# 发牌干预局数/人数、发牌干预占比、发牌干预被奖励者胜率
# 疑似伙牌用户数、同IP用户数、异常高胜率用户数

query = """
with collusion_player as(
select distinct dt,player1_id as playerid
from poker.dws_dz_player_collusion_cross_df 
where dt='${dt}'
and total_games>=5
and shared_ip_cnts is not null
and win_rate>=0.5
and (player1_same_rate>=0.5 or player2_same_rate>=0.5)
union all
select distinct dt,player2_id as playerid
from poker.dws_dz_player_collusion_cross_df 
where dt='${dt}'
and total_games>=5
and shared_ip_cnts is not null
and win_rate>=0.5
and (player1_same_rate>=0.5 or player2_same_rate>=0.5)
)


,same_ip_player as(
select distinct dt,player1_id as playerid
from poker.dws_dz_player_collusion_cross_df 
where dt='${dt}'
and shared_ip_cnts is not null
union all
select distinct dt,player2_id as playerid
from poker.dws_dz_player_collusion_cross_df 
where dt='${dt}'
and shared_ip_cnts is not null
)

,abnormal_high_win_player as(
select * from(
select dt,nplayerid
    ,count(distinct concat(sztoken,nround) ) as round_cnt
    ,count(distinct case when gameresult>0 then concat(sztoken,nround) end ) as win_round_cnt
    ,count(distinct case when gameresult>0 then concat(sztoken,nround) end )/count(distinct concat(sztoken,nround) )  as win_rate
from poker.dws_db_player_game_result_di
where  dt='${dt}'
group by dt,nplayerid)t
where win_rate>=0.8
and round_cnt>=10)


,total_cnt as(
select dt 
      ,count(distinct concat(sztoken,nround)) as round_cnt
      ,count(distinct case when brobot=1 then concat(sztoken,nround) end) as robot_round_cnt
      ,count(distinct case when brobot=1 then concat(sztoken,nround) end)/count(1) as robot_round_pct
      ,count(distinct case when brobot=1 and gameresult>=0 then concat(sztoken,nround) end) as robot_win_round_cnt
      ,count(distinct case when brobot=1 and gameresult>=0 then concat(sztoken,nround) end)/count(case when brobot=1 then sztoken end) as robot_win_round_pct
      ,count(distinct case when brobot=0 then nplayerid else null end) as day_real_player_cnt
from poker.dws_db_player_game_result_di 
where dt='${dt}'
group by dt)

-- MAU
,month_total_cnt as(
select dt 
      ,count(distinct case when brobot=0 then nplayerid else null end) as month_real_player_cnt
from poker.dws_db_player_game_result_di 
where dt<='${dt}' and dt>=date_sub('${dt}',30)
group by dt)

-- 当日新增用户数
,new_user_cnt as(
select t1.dt
  ,count(distinct case when t2.nplayerid is null then t1.nplayerid end) as new_player_cnt
from 
(select distinct dt,nplayerid 
from poker.ods_dz_db_table_user_df
where dt='${dt}'
and brobot='0') t1
left join 
(select distinct nplayerid
from poker.ods_dz_db_table_user_df
where dt=date_sub('${dt}',1)
and brobot='0') t2
on t1.nplayerid=t2.nplayerid
group by t1.dt
)

-- 发牌干预
,preset_cnt as(
select t1.dt
      ,count(distinct concat(t1.sztoken,t1.nround)) as preset_round_cnt
      ,count(distinct t1.nplayerid) as preset_player_cnt
      ,count(distinct case when presettype=2 then t1.nplayerid end) as reward_player_cnt
      ,count(distinct case when presettype=2 and gameresult>0 then t1.nplayerid end) as reward_win_round_cnt
      -- ,count(distinct case when presettype=2 then nplayerid end) as preset_reward_player_cnt
      -- ,count(distinct concat(t1.sztoken,t1.nround)) as preset_round_cnt
from poker.ods_card_preset_di t1
left join poker.dws_db_player_game_result_di t2
on t1.sztoken=t2.sztoken
and t1.nround=t2.nround
and t1.nplayerid=t2.nplayerid
and t1.dt=t2.dt
where bpreset='true'
and t1.dt='${dt}'
group by t1.dt
)


select t1.*
      ,t2.month_real_player_cnt
      ,t3.new_player_cnt
      ,coalesce(preset_round_cnt,0) as preset_round_cnt      
      ,coalesce(preset_player_cnt,0) as preset_player_cnt      
      ,coalesce(reward_player_cnt,0) as reward_player_cnt      
      ,coalesce(reward_win_round_cnt,0) as reward_win_round_cnt      
      ,coalesce(collusion_player_cnt,0) as collusion_player_cnt      
      ,coalesce(same_ip_player_cnt,0) as same_ip_player_cnt      
      ,coalesce(abnormal_high_win_player_cnt,0) as abnormal_high_win_player_cnt
from total_cnt t1
left join month_total_cnt t2
on t1.dt=t2.dt
left join new_user_cnt t3
on t1.dt=t3.dt
left join preset_cnt t4
on t1.dt=t4.dt
left join 
(select dt,count(distinct playerid) as collusion_player_cnt
from collusion_player
group by dt) t5
on t1.dt=t5.dt
left join 
(select dt,count(distinct playerid) as same_ip_player_cnt
from same_ip_player
group by dt) t6
on t1.dt=t6.dt
left join 
(select dt,count(distinct nplayerid) as
abnormal_high_win_player_cnt
from abnormal_high_win_player
group by dt) t7
on t1.dt=t7.dt

"""
query = query.replace('${dt}', dt_str)
cursor.execute(query)

# 获取数据并转换为 DataFrame
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=column_names)

# 关闭连接
cursor.close()
conn.close()

# 显示数据
print(df)

Querying for date: 2026-01-09
        t1.dt  t1.round_cnt  t1.robot_round_cnt  t1.robot_round_pct  \
0  2026-01-09         13566                9400            0.186264   

   t1.robot_win_round_cnt  t1.robot_win_round_pct  t1.day_real_player_cnt  \
0                    6270                 0.31949                    1008   

   t2.month_real_player_cnt  t3.new_player_cnt preset_round_cnt  \
0                      1008                646             None   

  preset_player_cnt reward_player_cnt reward_win_round_cnt  \
0              None              None                 None   

  collusion_player_cnt same_ip_player_cnt  abnormal_high_win_player_cnt  
0                 None               None                             2  
